## 환경 설정 및 라이브러리 임포트

In [17]:
import os
import csv
import win32com.client as win32
from pdf2image import convert_from_path

## 전역 변수 및 경로 설정

In [12]:
# 기준 디렉토리 설정
BASE_DIR = "../../data/ocr/prescriptions"

# 입력 파일 경로
TEMPLATE_PATH = os.path.join(BASE_DIR, "처방전_양식.hwp")
CSV_PATH = os.path.join(BASE_DIR, "처방전_데이터.csv")

# 출력 폴더 경로
OUTPUT_HWP_DIR = os.path.join(BASE_DIR, "output")
OUTPUT_PNG_DIR = os.path.join(BASE_DIR, "output_png")

# 출력 폴더 자동 생성
os.makedirs(OUTPUT_HWP_DIR, exist_ok=True)
os.makedirs(OUTPUT_PNG_DIR, exist_ok=True)

## HWP 제어 로직

In [13]:
class HWPProcessor:
    def __init__(self, visible=False):
        """HWP COM 객체를 초기화하고 백그라운드 실행 여부를 설정합니다."""
        try:
            self.hwp = win32.gencache.EnsureDispatch("HWPFrame.HwpObject")
            self.hwp.XHwpWindows.Item(0).Visible = visible
        except Exception as e:
            raise RuntimeError(f"HWP 연동 실패: {e}")

    def open(self, file_path):
        """절대 경로로 파일을 엽니다."""
        abs_path = os.path.abspath(file_path)
        if not os.path.exists(abs_path):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {abs_path}")
        self.hwp.Open(abs_path)

    def extract_fields(self):
        """현재 열린 문서의 누름틀 리스트를 반환합니다."""
        field_string = self.hwp.GetFieldList(1)
        if not field_string:
            return []
        return field_string.split('\x02')

    def fill_data(self, data: dict, append_suffix=False):
        """데이터를 누름틀에 입력합니다.
        append_suffix가 True이면 키 값 뒤에 '{{0}}'을 자동으로 붙입니다.
        """
        for field_name, value in data.items():
            target_field = f"{field_name}{{{{0}}}}" if append_suffix else field_name
            self.hwp.PutFieldText(target_field, str(value))

    def save_as(self, output_path, format_type="HWP"):
        """문서를 지정된 포맷으로 저장합니다."""
        self.hwp.SaveAs(os.path.abspath(output_path), format_type, "")

    def clear(self):
        """현재 문서를 닫고 초기화합니다."""
        self.hwp.Clear(1)

    def quit(self):
        """HWP 프로세스를 종료합니다."""
        self.hwp.Quit()


def batch_generate_hwp(processor, template_path, output_dir, csv_path):
    """CSV 데이터를 읽어 HWP 파일을 일괄 생성합니다."""
    with open(csv_path, encoding="utf-8-sig") as f:
        records = list(csv.DictReader(f))

    for i, record in enumerate(records):
        processor.open(template_path)
        processor.fill_data(record, append_suffix=True)

        output_path = os.path.join(output_dir, f"처방전_{i+1:04d}.hwp")
        processor.save_as(output_path, "HWP")
        processor.clear()
        print(f"저장 완료: {output_path}")

    print(f"총 {len(records)}개 HWP 생성 완료")


def batch_convert_to_png(processor, hwp_dir, output_dir, dpi=200):
    """디렉토리 내의 HWP 파일을 PDF로 임시 변환 후 PNG로 일괄 변환합니다."""
    hwp_files = sorted([f for f in os.listdir(hwp_dir) if f.endswith(".hwp")])

    for hwp_file in hwp_files:
        hwp_path = os.path.join(hwp_dir, hwp_file)
        pdf_path = hwp_path.replace(".hwp", ".pdf")
        base_name = os.path.splitext(hwp_file)[0]

        # 1. HWP -> PDF 변환
        processor.open(hwp_path)
        processor.save_as(pdf_path, "PDF")
        processor.clear()

        # 2. PDF -> PNG 변환
        images = convert_from_path(pdf_path, dpi=dpi)
        for j, img in enumerate(images):
            # 페이지가 여러 장일 경우 번호 부여, 한 장이면 파일명 그대로 사용
            suffix = f"_p{j+1}" if len(images) > 1 else ""
            png_path = os.path.join(output_dir, f"{base_name}{suffix}.png")
            img.save(png_path, "PNG")
            print(f"저장: {png_path}")

        # 3. 임시 PDF 파일 삭제
        os.remove(pdf_path)

    print(f"총 {len(hwp_files)}개 PNG 변환 완료")

## HWP 프로세서 초기화

In [14]:
# 프로세서 인스턴스 생성 (백그라운드 실행)
processor = HWPProcessor(visible=False)

## 누름틀 개수 확인

In [15]:
processor.open(TEMPLATE_PATH)
fields = processor.extract_fields()

print(f"총 누름틀 개수: {len(fields)}개")

processor.clear()

총 누름틀 개수: 62개


## 단건 데이터 테스트 생성

In [16]:
sample_data = {
    "facility_code{{0}}": "12345678",
    "facility_name{{0}}": "오즈코딩의원"
}

processor.open(TEMPLATE_PATH)
processor.fill_data(sample_data, append_suffix=False)

test_output = os.path.join(OUTPUT_HWP_DIR, "처방전_test.hwp")
processor.save_as(test_output)
processor.clear()

print(f"단건 테스트 저장 완료: {test_output}")

단건 테스트 저장 완료: ../../data/ocr/prescriptions\output\처방전_test.hwp


## CSV 기반 대량 생성

In [ ]:
# batch_generate_hwp(processor, TEMPLATE_PATH, OUTPUT_HWP_DIR, CSV_PATH)

## 생성된 HWP를 PNG로 일괄 변환

In [ ]:
# batch_convert_to_png(processor, OUTPUT_HWP_DIR, OUTPUT_PNG_DIR)

## 작업 종료 및 리소스 정리

In [18]:
processor.quit()
print("HWP 프로세스 종료 완료")

HWP 프로세스 종료 완료
